In [0]:
%run ../read_params

In [0]:
%run ../utils

In [0]:
%run ./schemas

In [0]:
def check_existence(table_name, col_name):
    """
    Doc String
    """
    existing_df = spark.table(f'{LOOKUPS_DATABASE_PREFIX}.{table_name}').select(col_name)

    existing_list = [row[col_name] for row in existing_df.collect()]

    return existing_list


In [0]:
def api_extraction(url, schema, base_dict = None):
    """
    Doc String
    """
    s_start_time = time.time()

    response = requests.get(url)
    data = response.json()

    s_end_time = time.time()
    time.sleep(s_end_time - s_start_time)

    if not base_dict is None:
        data = base_dict | data

    df = spark.createDataFrame([data], schema=schema)

    return df

In [0]:
def species_extraction(url, schema):
    """
    Doc String
    """
    species_df = api_extraction(url, schema)

    species_df.write.format('delta').mode("append").option('mergeSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.species")

    return species_df

In [0]:
def varieties_extraction(species_df):
    """
    Doc String
    """
    varieties = species_df.select('varieties').collect()[0][0]

    varieties_list = [[row['is_default'], row['pokemon']['name'], row['pokemon']['url']] for row in varieties]

    for is_default, variety_name, varieties_url in varieties_list:
        base_dict = {
            'base_pokedex_number' : pokedex_no,
            'base_name' : name
        }

        varieties_df = api_extraction(varieties_url, varieties_schema, base_dict)

        varieties_df.write.format('delta').mode("append").option('mergeSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.varieties")

    return varieties_df

In [0]:
def form_data_extraction(varieties_df):
    """
    Doc String
    """
    form_varieties = (varieties_df
                      .withColumn('form', explode('forms'))
                      .select(
                        'base_pokedex_number', 
                        'base_name', 
                        'id', 
                        'name', 
                        col('form.name').alias('form_name'),
                        col('form.url').alias('form_url')
                      )
    )

    form_urls = [[row['id'], row['form_url']] for row in form_varieties.collect()]

    for variety_id, form_url in form_urls:
        base_dict = {
            'pokeapi_id' : variety_id
        }

        form_df = api_extraction(form_url, form_schema, base_dict)

        form_df.write.format('delta').mode("append").option('mergeSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.forms")

    return form_df

In [0]:
def obtain_pokemon_extraction_list(lookup_table_name):
    pokemon_entries = spark.sql(f"""
        SELECT
            pokemon_entries 
        FROM 
            {STAGING_DATABASE_PREFIX}.nat_dex
    """).collect()[0][0]

    pokemon_entries_list_total = [[row['entry_number'], row['pokemon_species']['name'], row['pokemon_species']['url']] for row in pokemon_entries]

    existing_pokedex_list = check_existence(lookup_table_name, 'pokedex_no')

    pokemon_entries_list = [entry for entry in pokemon_entries_list_total if entry[0] not in existing_pokedex_list]

    total_entries = len(pokemon_entries_list)

    return pokemon_entries_list, total_entries

In [0]:
def obtain_pokemon_url_list(lookup_table_name):
    evolution_chain_urls = spark.sql(f"""
        SELECT DISTINCT 
            evolution_chain.url AS url
        FROM 
            {STAGING_DATABASE_PREFIX}.species
    """)

    evolution_chain_url_list_total = [row['url'] for row in evolution_chain_urls.collect()]

    existing_url_list = check_existence(lookup_table_name, 'url')

    evo_chain_url_list = [url for url in evolution_chain_url_list_total if url not in existing_url_list]

    total_urls = len(evo_chain_url_list)

    return evo_chain_url_list, total_urls

In [0]:
LOOKUPS_TABLE_NAME = 'pokedex_updated'

nat_dex_url = f'https://pokeapi.co/api/v2/pokedex/1/'
nat_dex_df = api_extraction(nat_dex_url, nat_dex_schema)

nat_dex_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.nat_dex")

pokemon_entries_list, total_entries = obtain_pokemon_extraction_list(LOOKUPS_TABLE_NAME)

num_entries_complete = 0

for pokedex_no, name, species_url in pokemon_entries_list:
    print(f"Starting {pokedex_no}: {name}")

    species_df = species_extraction(species_url, species_schema)

    varieties_df = varieties_extraction(species_df)

    form_df = form_data_extraction(varieties_df)

    insert_query = f"""
        INSERT INTO {LOOKUPS_DATABASE_PREFIX}.pokedex_updated
        VALUES ({pokedex_no}, '{name}', '{datetime.now(timezone.utc).replace(tzinfo = None)}')
    """
    spark.sql(insert_query)

    num_entries_complete += 1

    print(f"Completed {pokedex_no}: {name} - {num_entries_complete}/{total_entries}")

In [0]:
LOOKUPS_TABLE_NAME = 'pokedex_updated'

evolution_chain_url_list, total_urls = obtain_pokemon_url_list(LOOKUPS_TABLE_NAME)


for i, url in enumerate(evolution_chain_url_list):
    print(f'Starting {url}...')

    evolution_chain_df = api_extraction(url, evo_chain_schema)

    evolution_chain_df.write.format('delta').mode("append").option('mergeSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.evo_chain")

    print(f'Completed {url} - {i+1}/{total_urls}')

    insert_query = f"""
        INSERT INTO {LOOKUPS_DATABASE_PREFIX}.evo_chain_updated
        VALUES ('{url}', '{datetime.now(timezone.utc).replace(tzinfo = None)}')
    """
    spark.sql(insert_query)